In [1]:
import pandas as pd
import numpy as np
import re

from unidecode import unidecode
from rapidfuzz.fuzz import ratio, token_sort_ratio, token_set_ratio

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import NearestNeighbors

print("Libraries loaded successfully!")

Libraries loaded successfully!


In [2]:
train_s1 = pd.read_csv(
    "dataset/train/train_source1.tsv",
    sep="\t"
)

train_s2 = pd.read_csv(
    "dataset/train/train_source2.tsv",
    sep="\t"
)

train_s3 = pd.read_csv(
    "dataset/train/train_source3.tsv",
    sep="\t"
)

truth = pd.read_csv(
    "dataset/train/train_ground_truth.tsv",
    sep="\t",
    keep_default_na=False
)

print("Source 1:", train_s1.shape)
print("Source 2:", train_s2.shape)
print("Source 3:", train_s3.shape)
print("Ground truth:", truth.shape)

Source 1: (2206821, 4)
Source 2: (5034616, 4)
Source 3: (5285603, 4)
Ground truth: (2206821, 2)


In [3]:
def normalize_text(value):
    if pd.isna(value):
        return ""

    value = str(value)
    value = unidecode(value)
    value = value.lower()
    value = re.sub(r"[^a-z0-9\s]", " ", value)
    value = re.sub(r"\s+", " ", value).strip()

    return value


for df in [train_s1, train_s2, train_s3]:
    df["name_normalized"] = df["business_name"].apply(normalize_text)
    df["address_normalized"] = df["business_address"].apply(normalize_text)

print(train_s1[
    ["business_name", "name_normalized",
     "business_address", "address_normalized"]
].head())

             business_name          name_normalized  \
0      Orelee's Barbershop      orelee s barbershop   
1              Prime Money              prime money   
2            B+ Retail Inc             b retail inc   
3            Christ Chapel            christ chapel   
4  Prabhav Business Center  prabhav business center   

                                    business_address  \
0             1795 Westchester Drive, High Point, NC   
1                    17560 Ellis Road, Tahlequah, OK   
2                1712 Montebello Avenue, Phoenix, AZ   
3  2100 Cameron Drive, Unit APARTMENT G, Dundalk, MD   
4  797, Lake Town Block A, Kolkata, Howrah, West ...   

                                 address_normalized  
0              1795 westchester drive high point nc  
1                     17560 ellis road tahlequah ok  
2                 1712 montebello avenue phoenix az  
3    2100 cameron drive unit apartment g dundalk md  
4  797 lake town block a kolkata howrah west bengal  


In [4]:
# Step 6: Test TF-IDF blocking on a small sample

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import NearestNeighbors

# Small samples for testing
S1_SAMPLE_SIZE = 1000
CANDIDATE_SAMPLE_SIZE = 5000

s1_sample = train_s1.head(S1_SAMPLE_SIZE).copy()

# Combine S2 and S3 for candidate generation
s2_sample = train_s2.head(CANDIDATE_SAMPLE_SIZE // 2).copy()
s3_sample = train_s3.head(CANDIDATE_SAMPLE_SIZE // 2).copy()

candidate_sample = pd.concat(
    [s2_sample, s3_sample],
    ignore_index=True
)

# Use normalized business names
s1_names = s1_sample["name_normalized"].fillna("")
candidate_names = candidate_sample["name_normalized"].fillna("")

# Create TF-IDF character features
vectorizer = TfidfVectorizer(
    analyzer="char",
    ngram_range=(3, 5),
    min_df=2,
    dtype=np.float32
)

candidate_matrix = vectorizer.fit_transform(candidate_names)
s1_matrix = vectorizer.transform(s1_names)

print("S1 matrix:", s1_matrix.shape)
print("Candidate matrix:", candidate_matrix.shape)

# Find nearest candidates
knn = NearestNeighbors(
    n_neighbors=30,
    metric="cosine",
    algorithm="brute"
)

knn.fit(candidate_matrix)

distances, indices = knn.kneighbors(s1_matrix)

print("\nBlocking completed!")
print("Number of S1 records:", len(s1_sample))
print("Candidates per S1 record:", indices.shape[1])

S1 matrix: (1000, 25277)
Candidate matrix: (5000, 25277)

Blocking completed!
Number of S1 records: 1000
Candidates per S1 record: 30


In [6]:
# Step 6: Test TF-IDF blocking on a small sample

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import NearestNeighbors

S1_SAMPLE_SIZE = 1000
CANDIDATE_SAMPLE_SIZE = 5000

In [7]:
# Step 7: Evaluate blocking recall on the sample

# Create a mapping from entity_id -> row position
candidate_id_to_index = {
    entity_id: idx
    for idx, entity_id in enumerate(candidate_sample["entity_id"])
}

# Store candidate entity IDs for each S1 record
candidate_ids = candidate_sample.iloc[indices.ravel()]["entity_id"].to_numpy()
candidate_ids = candidate_ids.reshape(indices.shape)

# Create ground-truth lookup
truth_lookup = dict(
    zip(
        truth["source1_entity_id"],
        truth["matched_entity_ids"]
    )
)

hits = 0
evaluated = 0

for row_idx, s1_id in enumerate(s1_sample["entity_id"]):

    true_matches = truth_lookup.get(s1_id, "")

    if not true_matches:
        continue

    true_ids = set(
        true_matches.split(",")
    )

    generated_ids = set(
        candidate_ids[row_idx]
    )

    if true_ids.intersection(generated_ids):
        hits += 1

    evaluated += 1


recall = hits / evaluated if evaluated else 0

print("===== BLOCKING RECALL =====")
print("S1 records evaluated:", evaluated)
print("S1 records with at least one true match in candidates:", hits)
print("Blocking recall:", round(recall * 100, 2), "%")

===== BLOCKING RECALL =====
S1 records evaluated: 946
S1 records with at least one true match in candidates: 1
Blocking recall: 0.11 %


In [8]:
# Step 7A: Representative blocking recall test

import random

# Number of S1 records to evaluate
EVAL_SIZE = 1000

# Number of random negative candidates
RANDOM_NEGATIVES = 20000

# ---------------------------------------------------------
# 1. Select S1 records for evaluation
# ---------------------------------------------------------

eval_s1 = train_s1.head(EVAL_SIZE).copy()

eval_s1_ids = set(eval_s1["entity_id"])


# ---------------------------------------------------------
# 2. Get the ground-truth matched IDs
# ---------------------------------------------------------

eval_truth = truth[
    truth["source1_entity_id"].isin(eval_s1_ids)
].copy()

true_match_ids = set()

for value in eval_truth["matched_entity_ids"]:
    if value:
        true_match_ids.update(value.split(","))

print("Unique true match IDs:", len(true_match_ids))


# ---------------------------------------------------------
# 3. Retrieve the actual matched records from S2 and S3
# ---------------------------------------------------------

true_s2 = train_s2[
    train_s2["entity_id"].isin(true_match_ids)
].copy()

true_s3 = train_s3[
    train_s3["entity_id"].isin(true_match_ids)
].copy()

true_candidates = pd.concat(
    [true_s2, true_s3],
    ignore_index=True
)

print("True S2 matches found:", len(true_s2))
print("True S3 matches found:", len(true_s3))


# ---------------------------------------------------------
# 4. Add random negative candidates
# ---------------------------------------------------------

random_s2 = train_s2.sample(
    n=RANDOM_NEGATIVES // 2,
    random_state=42
)

random_s3 = train_s3.sample(
    n=RANDOM_NEGATIVES // 2,
    random_state=42
)

random_candidates = pd.concat(
    [random_s2, random_s3],
    ignore_index=True
)

# Remove accidental true matches from the random pool
random_candidates = random_candidates[
    ~random_candidates["entity_id"].isin(true_match_ids)
]

candidate_eval = pd.concat(
    [true_candidates, random_candidates],
    ignore_index=True
).drop_duplicates(
    subset=["entity_id"]
).reset_index(drop=True)


print("Total evaluation candidates:", len(candidate_eval))


# ---------------------------------------------------------
# 5. TF-IDF vectorization
# ---------------------------------------------------------

vectorizer_eval = TfidfVectorizer(
    analyzer="char",
    ngram_range=(3, 5),
    min_df=2,
    dtype=np.float32
)

candidate_matrix_eval = vectorizer_eval.fit_transform(
    candidate_eval["name_normalized"].fillna("")
)

s1_matrix_eval = vectorizer_eval.transform(
    eval_s1["name_normalized"].fillna("")
)

print("S1 matrix:", s1_matrix_eval.shape)
print("Candidate matrix:", candidate_matrix_eval.shape)


# ---------------------------------------------------------
# 6. Find top 30 candidates
# ---------------------------------------------------------

knn_eval = NearestNeighbors(
    n_neighbors=30,
    metric="cosine",
    algorithm="brute"
)

knn_eval.fit(candidate_matrix_eval)

distances_eval, indices_eval = knn_eval.kneighbors(
    s1_matrix_eval
)


# ---------------------------------------------------------
# 7. Calculate recall@30
# ---------------------------------------------------------

candidate_ids_eval = candidate_eval.iloc[
    indices_eval.ravel()
]["entity_id"].to_numpy()

candidate_ids_eval = candidate_ids_eval.reshape(
    indices_eval.shape
)

truth_lookup_eval = dict(
    zip(
        truth["source1_entity_id"],
        truth["matched_entity_ids"]
    )
)

hits = 0
evaluated = 0

for row_idx, s1_id in enumerate(eval_s1["entity_id"]):

    matched_string = truth_lookup_eval.get(
        s1_id,
        ""
    )

    if not matched_string:
        continue

    true_ids = set(
        matched_string.split(",")
    )

    generated_ids = set(
        candidate_ids_eval[row_idx]
    )

    if true_ids.intersection(generated_ids):
        hits += 1

    evaluated += 1


recall_at_30 = (
    hits / evaluated
    if evaluated
    else 0
)

print("\n===== REPRESENTATIVE BLOCKING RECALL =====")
print("S1 records evaluated:", evaluated)
print("Records with a true match in top 30:", hits)
print(
    "Recall@30:",
    round(recall_at_30 * 100, 2),
    "%"
)

Unique true match IDs: 3517
True S2 matches found: 1706
True S3 matches found: 1811
Total evaluation candidates: 23513
S1 matrix: (1000, 68235)
Candidate matrix: (23513, 68235)

===== REPRESENTATIVE BLOCKING RECALL =====
S1 records evaluated: 946
Records with a true match in top 30: 918
Recall@30: 97.04 %
